# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

## 1.2 Функции

In [5]:
def evaluate_classification(y_test, y_pred_proba):
    """Оценивает результаты классификации"""
    # Кодируем строковые метки в числа
    le = LabelEncoder()
    y_test_encoded = le.fit_transform(y_test)
    
    # Предсказанные классы
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Метрики
    accuracy = accuracy_score(y_test_encoded, y_pred)
    f1 = f1_score(y_test_encoded, y_pred, average='weighted')
    
    # Для многоклассовой классификации
    auc_roc = roc_auc_score(y_test_encoded, y_pred_proba, multi_class='ovr', average='weighted')
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"AUC-ROC: {auc_roc:.4f}")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test_encoded, y_pred, target_names=['L (падение)', 'N (нейтр)', 'R (рост)']))

In [6]:
def train_predict_rf(X_train, X_test, y_train):
    """Обучает и предсказывает Random Forest с подбором гиперпараметров"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    
    model = RandomForestClassifier(random_state=13)
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
    grid_search.fit(X_train, y_train_encoded)
    
    print(f"Random Forest - лучшие параметры: {grid_search.best_params_}")
    
    y_pred_proba = grid_search.predict_proba(X_test)
    return y_pred_proba

def train_predict_dt(X_train, X_test, y_train):
    """Обучает и предсказывает Decision Tree с подбором гиперпараметров"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    param_grid = {
        'max_depth': [None, 5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']
    }
    
    model = DecisionTreeClassifier(random_state=13)
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
    grid_search.fit(X_train, y_train_encoded)
    
    print(f"Decision Tree - лучшие параметры: {grid_search.best_params_}")
    
    y_pred_proba = grid_search.predict_proba(X_test)
    return y_pred_proba

def train_predict_logreg(X_train, X_test, y_train):
    """Обучает и предсказывает Logistic Regression с подбором гиперпараметров"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    param_grid = {
        'C': [0.1, 1, 10, 100],
        'solver': ['lbfgs', 'liblinear', 'saga'],
        'penalty': ['l2', 'l1']
    }
    
    model = LogisticRegression(max_iter=1000, random_state=13)
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
    grid_search.fit(X_train, y_train_encoded)
    
    print(f"Logistic Regression - лучшие параметры: {grid_search.best_params_}")
    
    y_pred_proba = grid_search.predict_proba(X_test)
    return y_pred_proba

def train_predict_catboost(X_train, X_test, y_train):
    """Обучает и предсказывает CatBoost с подбором гиперпараметров"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    param_grid = {
        'depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'l2_leaf_reg': [1, 3, 5],
        'iterations': [100, 200, 500]
    }
    
    model = CatBoostClassifier(verbose=False, random_state=13)
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1)  # cv=3 для скорости
    grid_search.fit(X_train, y_train_encoded)
    
    print(f"CatBoost - лучшие параметры: {grid_search.best_params_}")
    
    y_pred_proba = grid_search.predict_proba(X_test)
    return y_pred_proba

In [7]:
def train_test_split_by_date(df, target_column, test_size=0.2):
    """
    Разбивает данные на train/test по дате и возвращает X, y
    
    Args:
        df: DataFrame с колонкой 'begin'
        target_column: название целевой переменной
        test_size: доля тестовых данных (0.2 = 20%)
    """
    df = df.sort_values('begin').reset_index(drop=True)
    
    # Вычисляем индекс разбиения
    split_idx = int(len(df) * (1 - test_size))
    
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    print(f"Train: {train_df['begin'].min()} - {train_df['begin'].max()} ({len(train_df)} samples)")
    print(f"Test:  {test_df['begin'].min()} - {test_df['begin'].max()} ({len(test_df)} samples)")
    
    # Удаляем колонку 'begin' и разделяем на X, y
    X_train = train_df.drop(columns=['begin', target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=['begin', target_column])
    y_test = test_df[target_column]
    
    print(f"Признаков: {X_train.shape[1]}")
    
    return X_train, X_test, y_train, y_test

# 2 Подготовка данных

## 2.0 Список тикеров

In [10]:
tickers = [
    'SBER', 'TCSG', 'GAZP', 'LKOH', 'ROSN'
]

## 2.1 Чтение

In [17]:
data_SBER = pd.read_csv("../../../data/stock_features_data/stocks_features_SBER.csv")
data_TCSG = pd.read_csv("../../../data/stock_features_data/stocks_features_TCSG.csv")
data_GAZP = pd.read_csv("../../../data/stock_features_data/stocks_features_GAZP.csv")
data_LKOH = pd.read_csv("../../../data/stock_features_data/stocks_features_LKOH.csv")
data_ROSN = pd.read_csv("../../../data/stock_features_data/stocks_features_ROSN.csv")
data_ROSN.head(2)

,begin,close,MA_90,RSI_14,RSI_90,MOM_10,ATR_14,VOLATILITY_20,VOLATILITY_50,VOLUME_RATIO_20,MACD_SIGNAL,MACD_HISTOGRAM,target_price_change,target_class
0,2022-05-31 18:00:00,377.00,385.952778,41.700213,46.109636,-6.102117,6.815141,0.177819,0.184228,0.180549,1.622548,-2.115281,-2.771883,L
1,2022-06-01 10:00:00,378.35,385.708333,43.536971,46.522581,-3.703232,6.960488,0.153935,0.177697,0.770980,1.125999,-1.986196,-1.361174,L


## 2.2 Удаление лишних признаков

In [20]:
# Для SBER
part_SBER = data_SBER[['begin', 'close', 'target_price_change']]
data_SBER.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для TCSG
part_TCSG = data_TCSG[['begin', 'close', 'target_price_change']]
data_TCSG.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для GAZP
part_GAZP = data_GAZP[['begin', 'close', 'target_price_change']]
data_GAZP.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для LKOH
part_LKOH = data_LKOH[['begin', 'close', 'target_price_change']]
data_LKOH.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для ROSN
part_ROSN = data_ROSN[['begin', 'close', 'target_price_change']]
data_ROSN.drop(['close', 'target_price_change'], axis=1, inplace=True)

## 2.3 Разделение на train/test

In [26]:
X_train_SBER, X_test_SBER, y_train_SBER, y_test_SBER = train_test_split_by_date(
    df=data_SBER, 
    target_column='target_class',
    test_size=0.2
)

X_train_TCSG, X_test_TCSG, y_train_TCSG, y_test_TCSG = train_test_split_by_date(
    df=data_TCSG, 
    target_column='target_class',
    test_size=0.2
)

X_train_GAZP, X_test_GAZP, y_train_GAZP, y_test_GAZP = train_test_split_by_date(
    df=data_GAZP, 
    target_column='target_class',
    test_size=0.2
)

X_train_LKOH, X_test_LKOH, y_train_LKOH, y_test_LKOH = train_test_split_by_date(
    df=data_LKOH, 
    target_column='target_class',
    test_size=0.2
)

X_train_ROSN, X_test_ROSN, y_train_ROSN, y_test_ROSN = train_test_split_by_date(
    df=data_ROSN, 
    target_column='target_class',
    test_size=0.2
)

Train: 2022-05-31 18:00:00 - 2025-03-11 14:00:00 (3427 samples)
Test:  2025-03-11 16:00:00 - 2025-09-30 18:00:00 (857 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2024-06-03 10:00:00 (2367 samples)
Test:  2024-06-03 12:00:00 - 2024-11-20 18:00:00 (592 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2025-03-11 14:00:00 (3427 samples)
Test:  2025-03-11 16:00:00 - 2025-09-30 18:00:00 (857 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2025-03-11 14:00:00 (3427 samples)
Test:  2025-03-11 16:00:00 - 2025-09-30 18:00:00 (857 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2025-03-11 14:00:00 (3427 samples)
Test:  2025-03-11 16:00:00 - 2025-09-30 18:00:00 (857 samples)
Признаков: 10


# 3 Обучение моделей с подбором гиперпараметров

## 3.1 Дерево решений

### 3.1.1 Сбер

In [20]:
y_pred_SBER = train_predict_dt(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Decision Tree - лучшие параметры: {'criterion': 'gini', 'max_depth': 15, 'min_samples_leaf': 4, 'min_samples_split': 2}
Accuracy: 0.4189
F1-score: 0.3755
AUC-ROC: 0.5746

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.84      0.51       269
   N (нейтр)       0.67      0.31      0.42       379
    R (рост)       0.28      0.08      0.13       209

    accuracy                           0.42       857
   macro avg       0.44      0.41      0.35       857
weighted avg       0.48      0.42      0.38       857



### 3.1.2 Тиньк

In [22]:
y_pred_TCSG = train_predict_dt(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Decision Tree - лучшие параметры: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2}
Accuracy: 0.3598
F1-score: 0.3516
AUC-ROC: 0.4807

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.30      0.50      0.38       202
   N (нейтр)       0.45      0.35      0.39       274
    R (рост)       0.35      0.16      0.22       116

    accuracy                           0.36       592
   macro avg       0.37      0.33      0.33       592
weighted avg       0.38      0.36      0.35       592



### 3.1.3 Газпром

In [24]:
y_pred_GAZP = train_predict_dt(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Decision Tree - лучшие параметры: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Accuracy: 0.3536
F1-score: 0.3657
AUC-ROC: 0.4833

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.32      0.28      0.30       285
   N (нейтр)       0.49      0.44      0.46       464
    R (рост)       0.11      0.19      0.14       108

    accuracy                           0.35       857
   macro avg       0.30      0.30      0.30       857
weighted avg       0.38      0.35      0.37       857



### 3.1.4 Лукойл

In [26]:
y_pred_LKOH = train_predict_dt(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Decision Tree - лучшие параметры: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2}
Accuracy: 0.2719
F1-score: 0.2200
AUC-ROC: 0.4778

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.27      0.65      0.38       270
   N (нейтр)       0.32      0.11      0.16       388
    R (рост)       0.21      0.08      0.11       199

    accuracy                           0.27       857
   macro avg       0.26      0.28      0.22       857
weighted avg       0.28      0.27      0.22       857



### 3.1.5 Роснефть

In [36]:
y_pred_ROSN = train_predict_dt(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Decision Tree - лучшие параметры: {'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Accuracy: 0.3862
F1-score: 0.3640
AUC-ROC: 0.5330

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.37      0.70      0.49       291
   N (нейтр)       0.57      0.27      0.36       415
    R (рост)       0.14      0.11      0.12       151

    accuracy                           0.39       857
   macro avg       0.36      0.36      0.33       857
weighted avg       0.43      0.39      0.36       857



## Вывод

Качество улучшилось по сравнению с вариантом без подбора гиперпараметров на всех бумагах

## 3.2 Регрессия

### 3.2.1 Сбер

In [41]:
y_pred_SBER = train_predict_logreg(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Logistic Regression - лучшие параметры: {'C': 1, 'penalty': 'l1', 'solver': 'saga'}
Accuracy: 0.3092
F1-score: 0.3036
AUC-ROC: 0.5033

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.29      0.32      0.31       269
   N (нейтр)       0.37      0.22      0.28       379
    R (рост)       0.28      0.45      0.35       209

    accuracy                           0.31       857
   macro avg       0.31      0.33      0.31       857
weighted avg       0.32      0.31      0.30       857



### 3.2.2 Тиньк

In [43]:
y_pred_TCSG = train_predict_logreg(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Logistic Regression - лучшие параметры: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}
Accuracy: 0.4189
F1-score: 0.3393
AUC-ROC: 0.5721

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.37      0.91      0.53       202
   N (нейтр)       0.64      0.23      0.34       274
    R (рост)       0.00      0.00      0.00       116

    accuracy                           0.42       592
   macro avg       0.34      0.38      0.29       592
weighted avg       0.42      0.42      0.34       592



### 3.2.3 Газпром

In [45]:
y_pred_GAZP = train_predict_logreg(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Logistic Regression - лучшие параметры: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}
Accuracy: 0.4399
F1-score: 0.4181
AUC-ROC: 0.5241

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.34      0.52      0.41       285
   N (нейтр)       0.55      0.49      0.52       464
    R (рост)       0.00      0.00      0.00       108

    accuracy                           0.44       857
   macro avg       0.30      0.34      0.31       857
weighted avg       0.41      0.44      0.42       857



### 3.2.4 Лукойл

In [47]:
y_pred_LKOH = train_predict_logreg(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Logistic Regression - лучшие параметры: {'C': 0.1, 'penalty': 'l1', 'solver': 'saga'}
Accuracy: 0.3781
F1-score: 0.3563
AUC-ROC: 0.5800

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.35      0.70      0.46       270
   N (нейтр)       0.60      0.29      0.40       388
    R (рост)       0.17      0.11      0.13       199

    accuracy                           0.38       857
   macro avg       0.37      0.37      0.33       857
weighted avg       0.42      0.38      0.36       857



### 3.2.5 Роснефть

In [49]:
y_pred_ROSN = train_predict_logreg(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Logistic Regression - лучшие параметры: {'C': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
Accuracy: 0.3967
F1-score: 0.3402
AUC-ROC: 0.5462

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.82      0.50       291
   N (нейтр)       0.59      0.22      0.32       415
    R (рост)       0.27      0.05      0.08       151

    accuracy                           0.40       857
   macro avg       0.40      0.37      0.30       857
weighted avg       0.45      0.40      0.34       857



## Вывод

Подбор гиперпараметров почти не повлиял на качество

## 3.3 Вот он, лес

### 3.3.1 Сбер

In [54]:
y_pred_SBER = train_predict_rf(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Random Forest - лучшие параметры: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}
Accuracy: 0.3687
F1-score: 0.3575
AUC-ROC: 0.5275

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.32      0.57      0.41       269
   N (нейтр)       0.52      0.36      0.43       379
    R (рост)       0.22      0.13      0.16       209

    accuracy                           0.37       857
   macro avg       0.36      0.35      0.33       857
weighted avg       0.39      0.37      0.36       857



### 3.3.2 Тиньк

In [56]:
y_pred_TCSG = train_predict_rf(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Random Forest - лучшие параметры: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 50}
Accuracy: 0.3784
F1-score: 0.3498
AUC-ROC: 0.5448

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.35      0.70      0.47       202
   N (нейтр)       0.59      0.21      0.31       274
    R (рост)       0.26      0.22      0.24       116

    accuracy                           0.38       592
   macro avg       0.40      0.38      0.34       592
weighted avg       0.45      0.38      0.35       592



### 3.3.3 Газпром

In [58]:
y_pred_GAZP = train_predict_rf(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Random Forest - лучшие параметры: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Accuracy: 0.3757
F1-score: 0.3678
AUC-ROC: 0.4349

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.30      0.46      0.37       285
   N (нейтр)       0.50      0.40      0.45       464
    R (рост)       0.07      0.03      0.04       108

    accuracy                           0.38       857
   macro avg       0.29      0.30      0.28       857
weighted avg       0.38      0.38      0.37       857



### 3.3.4 Лукойл

In [60]:
y_pred_LKOH = train_predict_rf(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Random Forest - лучшие параметры: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Accuracy: 0.3244
F1-score: 0.2382
AUC-ROC: 0.5022

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.32      0.84      0.46       270
   N (нейтр)       0.45      0.12      0.18       388
    R (рост)       0.14      0.03      0.04       199

    accuracy                           0.32       857
   macro avg       0.30      0.33      0.23       857
weighted avg       0.33      0.32      0.24       857



### 3.3.5 Роснефть

In [62]:
y_pred_ROSN = train_predict_rf(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Random Forest - лучшие параметры: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Accuracy: 0.3979
F1-score: 0.3515
AUC-ROC: 0.6073

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.80      0.50       291
   N (нейтр)       0.58      0.24      0.34       415
    R (рост)       0.22      0.07      0.11       151

    accuracy                           0.40       857
   macro avg       0.39      0.37      0.31       857
weighted avg       0.44      0.40      0.35       857



## Вывод

Качество немного улучшилось по сравнению с вариантом без подбора гиперпараметров, везде кроме роснефти

## 3.4 Бустинг

### 3.4.1 Сбер

In [67]:
y_pred_SBER = train_predict_catboost(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

CatBoost - лучшие параметры: {'depth': 6, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.1}
Accuracy: 0.3664
F1-score: 0.3479
AUC-ROC: 0.5334

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.62      0.45       269
   N (нейтр)       0.48      0.32      0.38       379
    R (рост)       0.19      0.12      0.15       209

    accuracy                           0.37       857
   macro avg       0.34      0.35      0.33       857
weighted avg       0.37      0.37      0.35       857



### 3.4.2 Тиньк

In [69]:
y_pred_TCSG = train_predict_catboost(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

CatBoost - лучшие параметры: {'depth': 8, 'iterations': 200, 'l2_leaf_reg': 5, 'learning_rate': 0.1}
Accuracy: 0.3767
F1-score: 0.3629
AUC-ROC: 0.5676

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.40      0.63      0.49       202
   N (нейтр)       0.59      0.22      0.32       274
    R (рост)       0.21      0.30      0.25       116

    accuracy                           0.38       592
   macro avg       0.40      0.38      0.35       592
weighted avg       0.45      0.38      0.36       592



### 3.4.3 Газпром

In [71]:
y_pred_GAZP = train_predict_catboost(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

CatBoost - лучшие параметры: {'depth': 8, 'iterations': 100, 'l2_leaf_reg': 3, 'learning_rate': 0.01}
Accuracy: 0.4049
F1-score: 0.3964
AUC-ROC: 0.4576

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.33      0.51      0.40       285
   N (нейтр)       0.55      0.43      0.49       464
    R (рост)       0.02      0.01      0.01       108

    accuracy                           0.40       857
   macro avg       0.30      0.32      0.30       857
weighted avg       0.41      0.40      0.40       857



### 3.4.4 Лукойл

In [73]:
y_pred_LKOH = train_predict_catboost(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

CatBoost - лучшие параметры: {'depth': 4, 'iterations': 500, 'l2_leaf_reg': 5, 'learning_rate': 0.05}
Accuracy: 0.3372
F1-score: 0.2652
AUC-ROC: 0.5719

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.31      0.81      0.45       270
   N (нейтр)       0.49      0.17      0.26       388
    R (рост)       0.27      0.02      0.04       199

    accuracy                           0.34       857
   macro avg       0.36      0.33      0.25       857
weighted avg       0.38      0.34      0.27       857



### 3.4.5 Роснефть

In [75]:
y_pred_ROSN = train_predict_catboost(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

CatBoost - лучшие параметры: {'depth': 4, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.1}
Accuracy: 0.3979
F1-score: 0.3576
AUC-ROC: 0.5964

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.77      0.49       291
   N (нейтр)       0.58      0.25      0.35       415
    R (рост)       0.23      0.08      0.12       151

    accuracy                           0.40       857
   macro avg       0.39      0.37      0.32       857
weighted avg       0.44      0.40      0.36       857



## Вывод

На всех акциях кроме Сбера качество предсказаний ухудшилось

Но в дальнейшем все равно хочеться использовать градиентный бустинг, так как это более мощный алгоритм